# AIDP - Sales Data MERGE Operation - Complete Example

## README: AIDP External Catalog - MERGE

This notebook demonstrates a complete MERGE operation workflow into ALH (Autonomous AI Lakehouse) database table using the AIDP framework.

### Key Concepts:

**Catalog User: `aidp_user`**
- `aidp_user` is a database user that is used to create the external catalog in AIDP
- This user has been granted access to the `coder` schema in database
- The `coder` schema contains tables that are managed by aidp_user

**Table Location in AIDP:**
- **Full Table Name**: `ext_private_catalog_aidp_user.coder.sales`
  - `ext_private_catalog_aidp_user`: External catalog reference for aidp_user
  - `coder`: Schema name
  - `sales`: Table name

### Table Schema and Create Statement:

**Please execute the following CREATE TABLE statement in your database (coder schema) before running this notebook:**

```sql
CREATE TABLE IF NOT EXISTS coder.sales (
    order_id NUMBER NOT NULL,
    customer_name VARCHAR2(100),
    product_name VARCHAR2(100),
    quantity NUMBER
)
```

**Column Descriptions:**
- `order_id` (NUMBER): Unique order identifier - Used as the identifier for MERGE operations
- `customer_name` (VARCHAR2): Name of the customer placing the order
- `product_name` (VARCHAR2): Name of the product ordered
- `quantity` (NUMBER): Quantity of the product ordered

### Workflow Overview:
1. Ensure the sales table exists in the coder schema (execute CREATE TABLE statement above)
2. Insert initial sample customer order records
3. Create an update/insert DataFrame (simulating new orders and updates)
4. Execute MERGE operation to sync the DataFrame with the table
5. Verify final results with summary statistics

### Merge Operation Details:
- **Merge Mode**: "MERGE" mode enabled
- **Merge Keys**: "order_id"
- **Skip object-storage Staging**: Enabled for performance optimization
- **Merge Staging Using Username**: Uses current session username (aidp_user) for staging table. If the user has a special case of requiring merge staging in the logged in user (aidp_user), then this flag can be set to true.

## Step 1: Define Target Table

In [11]:
# Define target table location
catalog_name = "ext_private_catalog_aidp_user"
schema_name = "coder"
table_name = "sales"
full_target_table = f"{catalog_name}.{schema_name}.{table_name}"

print(f"Target Table: {full_target_table}")
print(f"\nCatalog: {catalog_name}")
print(f"Schema: {schema_name}")
print(f"Table: {table_name}")
print("\n✓ Ensure the CREATE TABLE statement from README has been executed in your database before proceeding.")

Target Table: ext_private_catalog_aidp_user.coder.sales

Catalog: ext_private_catalog_aidp_user
Schema: coder
Table: sales

✓ Ensure the CREATE TABLE statement from README has been executed in your database before proceeding.


## Step 2: Create Initial DataFrame with Sample Data

In [12]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
import pyspark.sql.functions as F

# Define schema for sales data
schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("quantity", IntegerType(), True)
])

# Create initial test data - Starting inventory
initial_data = [
    (1001, "John Smith", "Laptop", 2),
    (1002, "Jane Doe", "Monitor", 4),
    (1003, "Bob Johnson", "Keyboard", 5),
    (1004, "Alice Brown", "Mouse", 10)
]

df_initial = spark.createDataFrame(initial_data, schema=schema)

print("\n" + "="*70)
print("=== INITIAL TEST DATA (New Records to Insert) ===")
print("="*70)
print(f"\nTotal records: {df_initial.count()}")
df_initial.show(truncate=False)


=== INITIAL TEST DATA (New Records to Insert) ===



Total records: 4
+--------+-------------+------------+--------+
|order_id|customer_name|product_name|quantity|
+--------+-------------+------------+--------+
|1001    |John Smith   |Laptop      |2       |
|1002    |Jane Doe     |Monitor     |4       |
|1003    |Bob Johnson  |Keyboard    |5       |
|1004    |Alice Brown  |Mouse       |10      |
+--------+-------------+------------+--------+



## Step 3: Insert Initial Data into Sales Table

In [13]:
# Insert initial data - Create temp view and use SQL INSERT
print("\n📝 Inserting initial sample data into target table...")

# Create temporary view from DataFrame
df_initial.createOrReplaceTempView("temp_sales_data")

# Insert using SQL with temp view
insert_sql = f"""
    INSERT INTO {full_target_table}
    SELECT * FROM temp_sales_data
"""

spark.sql(insert_sql)

print(f"✓ {df_initial.count()} records inserted into {full_target_table}")


📝 Inserting initial sample data into target table...


✓ 4 records inserted into ext_private_catalog_aidp_user.coder.sales


## Step 4: Verify Inserted Data

In [14]:
# Read the current data from the table
df_current = spark.read.table(full_target_table)

print("\n" + "="*70)
print("=== TABLE DATA AFTER INITIAL INSERT ===")
print("="*70)
print(f"\nTotal records: {df_current.count()}")
df_current.orderBy("order_id").show(truncate=False)


=== TABLE DATA AFTER INITIAL INSERT ===



Total records: 5


+---------------+-------------+------------+-------------+
|order_id       |customer_name|product_name|quantity     |
+---------------+-------------+------------+-------------+
|1.0000000000   |Test User    |Test Product|25.0000000000|
|1001.0000000000|John Smith   |Laptop      |2.0000000000 |
|1002.0000000000|Jane Doe     |Monitor     |4.0000000000 |
|1003.0000000000|Bob Johnson  |Keyboard    |5.0000000000 |
|1004.0000000000|Alice Brown  |Mouse       |10.0000000000|
+---------------+-------------+------------+-------------+



## Step 5: Create Update and Insert DataFrame for MERGE

In [15]:
# Create data for MERGE operation
# This includes:
# - Updated records (order_id: 1001, 1002) with quantity changes
# - New records (order_id: 1005, 1006) to be inserted

merge_data = [
    (1001, "John Smith", "Laptop Pro", 3),        # Update: Change product and quantity
    (1002, "Jane Doe", "Monitor 4K", 5),        # Update: Change product and quantity
    (1005, "David Wilson", "Headphones", 8),    # Insert: New order
    (1006, "Emma Davis", "Webcam", 3)           # Insert: New order
]

df_merge_data = spark.createDataFrame(merge_data, schema=schema)

print("\n" + "="*70)
print("=== DATA PREPARED FOR MERGE OPERATION ===")
print("="*70)
print("\nThis DataFrame contains:")
print("  - UPDATED records: order_id 1001, 1002 (quantity and product changes)")
print("  - NEW records: order_id 1005, 1006 (to be inserted)")
print(f"\nTotal records to merge: {df_merge_data.count()}")
df_merge_data.show(truncate=False)


=== DATA PREPARED FOR MERGE OPERATION ===

This DataFrame contains:
  - UPDATED records: order_id 1001, 1002 (quantity and product changes)
  - NEW records: order_id 1005, 1006 (to be inserted)



Total records to merge: 4


+--------+-------------+------------+--------+
|order_id|customer_name|product_name|quantity|
+--------+-------------+------------+--------+
|1001    |John Smith   |Laptop Pro  |3       |
|1002    |Jane Doe     |Monitor 4K  |5       |
|1005    |David Wilson |Headphones  |8       |
|1006    |Emma Davis   |Webcam      |3       |
+--------+-------------+------------+--------+



## Step 6: Execute MERGE Operation

In [ ]:
# Execute MERGE operation with AIDP-specific options
print("\n" + "="*70)
print("🔄 EXECUTING MERGE OPERATION")
print("="*70)
print(f"\nTarget Table: {full_target_table}")
print(f"Records to merge: {df_merge_data.count()}")
print("\nMERGE Options:")
print('  - write.mode: "MERGE"')
print('  - write.merge.keys: "order_id" (order_id specific key constraint)')
print('  - skip.oos.staging: "true" (skip out-of-sync staging)')
print('  - merge.staging.using.username: "true" (use session username)\n')

# Execute MERGE operation using DataFrame write API
df_merge_data.write \
    .option("write.mode", "MERGE") \
    .option("write.merge.keys", "order_id") \
    .option("skip.oos.staging", "true") \
    .option("merge.staging.using.username", "true") \
    .insertInto(full_target_table)

print("✓ MERGE operation completed successfully")
print("✓ Updated 2 records (order_id: 1001, 1002)")
print("✓ Inserted 2 new records (order_id: 1005, 1006)")


🔄 EXECUTING MERGE OPERATION

Target Table: ext_private_catalog_aidp_user.coder.sales


Records to merge: 4

MERGE Options:
  - write.mode: "MERGE"
  - write.merge.keys: "NO" (no specific key constraint)
  - skip.oos.staging: "true" (skip out-of-sync staging)
  - merge.staging.using.username: "true" (use session username)



✓ MERGE operation completed successfully
✓ Updated 2 records (order_id: 1001, 1002)
✓ Inserted 2 new records (order_id: 1005, 1006)


## Step 7: Verify MERGE Results

In [17]:
# Read the updated data from the table
df_after_merge = spark.read.table(full_target_table)

print("\n" + "="*70)
print("=== TABLE DATA AFTER MERGE OPERATION ===")
print("="*70)
print(f"\nTotal records: {df_after_merge.count()}")
print("\nExpected Results:")
print("  - order_id 1001: Product updated to 'Laptop Pro', quantity to 3")
print("  - order_id 1002: Product updated to 'Monitor 4K', quantity to 5")
print("  - order_id 1003: Keyboard (unchanged from initial insert)")
print("  - order_id 1004: Mouse (unchanged from initial insert)")
print("  - order_id 1005: NEW - David Wilson, Headphones, quantity 8")
print("  - order_id 1006: NEW - Emma Davis, Webcam, quantity 3")
print("\nActual Results:")
df_after_merge.orderBy("order_id").show(truncate=False)


=== TABLE DATA AFTER MERGE OPERATION ===



Total records: 7

Expected Results:
  - order_id 1001: Product updated to "Laptop Pro", quantity to 3
  - order_id 1002: Product updated to "Monitor 4K", quantity to 5
  - order_id 1003: Keyboard (unchanged from initial insert)
  - order_id 1004: Mouse (unchanged from initial insert)
  - order_id 1005: NEW - David Wilson, Headphones, quantity 8
  - order_id 1006: NEW - Emma Davis, Webcam, quantity 3

Actual Results:


+---------------+-------------+------------+-------------+
|order_id       |customer_name|product_name|quantity     |
+---------------+-------------+------------+-------------+
|1.0000000000   |Test User    |Test Product|25.0000000000|
|1001.0000000000|John Smith   |Laptop Pro  |3.0000000000 |
|1002.0000000000|Jane Doe     |Monitor 4K  |5.0000000000 |
|1003.0000000000|Bob Johnson  |Keyboard    |5.0000000000 |
|1004.0000000000|Alice Brown  |Mouse       |10.0000000000|
|1005.0000000000|David Wilson |Headphones  |8.0000000000 |
|1006.0000000000|Emma Davis   |Webcam      |3.0000000000 |
+---------------+-------------+------------+-------------+



## Key Takeaways

### Create Table - SQL Statement:
```sql
CREATE TABLE IF NOT EXISTS coder.sales (
    order_id NUMBER NOT NULL,
    customer_name VARCHAR2(100),
    product_name VARCHAR2(100),
    quantity NUMBER
)
```

### Initial Data Insert - Using Temporary View and SQL:
```python
# Create temporary view from DataFrame
df_initial.createOrReplaceTempView("temp_sales_data")

# Insert using SQL
insert_sql = f"""
    INSERT INTO {full_target_table}
    SELECT * FROM temp_sales_data
"""
spark.sql(insert_sql)
```

### MERGE Operation Behavior:
- **write.mode = "MERGE"**: Enables the MERGE operation mode for insert/update logic
- **write.merge.keys = "NO"**: No specific key constraint; all records are processed as-is
- **All records in the DataFrame will be inserted or updated into the target table**
- **Atomic Operation**: All INSERT and UPDATE operations happen as a single transaction

### AIDP-Specific Options:
- **skip.oos.staging**: Skips object-storage Staging for performance optimization
- **merge.staging.using.username**: Uses the current session's username for staging operations

### Spark DataFrame MERGE Syntax:
```python
df.write \
    .option("write.mode", "MERGE") \
    .option("write.merge.keys", "NO") \
    .option("skip.oos.staging", "true") \
    .option("merge.staging.using.username", "true") \
    .insertInto(full_target_table)
```

### Notebook Workflow:
- **Step 1**: Define Target Table - Sets up table reference
- **Step 2**: Create Initial DataFrame - Prepares sample data
- **Step 3**: INSERT Initial Data - Inserts records via SQL
- **Step 4**: Verify Inserted Data - Confirms insertion
- **Step 5**: Create MERGE DataFrame - Prepares update/insert data
- **Step 6**: Execute MERGE - Runs the MERGE operation
- **Step 7**: Verify MERGE Results - Validates final state
- **Step 8**: Summary Report - Generates statistics